<a href="https://colab.research.google.com/github/Gagangowdatg/BIS_1BM24CS098/blob/main/Grey_wolf_optamization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import f1_score, accuracy_score

# 1. LOAD & PREPROCESS DATA (Training & Validation Split)
data = load_breast_cancer()
X_train, X_val, y_train, y_val = train_test_split(
    data.data, data.target, test_size=0.3, random_state=42
)

# SVMs require feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

# Search Space Bounds for SVM: [C, gamma]
# Using log10 scale so GWO searches smoothly across small and large magnitudes:
# C from 10^-2 (0.01) to 10^3 (1000) -> [-2, 3]
# gamma from 10^-4 (0.0001) to 10^1 (10) -> [-4, 1]
LB = np.array([-2.0, -4.0])
UB = np.array([3.0, 1.0])
DIM = len(LB)

# Objective Function: Evaluate a wolf's hyperparameter position
def evaluate_fitness(wolf):
    C_val = 10 ** wolf[0]
    gamma_val = 10 ** wolf[1]

    model = SVC(C=C_val, gamma=gamma_val, kernel='rbf', random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)

    # Return validation F1-score (We want to MAXIMIZE this)
    return f1_score(y_val, preds)

# 2. INITIALIZE GWO PARAMETERS
N = 12          # Number of wolves in the pack
max_iter = 20   # Maximum iterations

# 3. INITIALIZE POPULATION (Random wolf positions)
wolves = np.random.uniform(LB, UB, size=(N, DIM))

# Initialize Alpha, Beta, and Delta wolves
alpha_pos, alpha_score = np.zeros(DIM), -float('inf')
beta_pos, beta_score = np.zeros(DIM), -float('inf')
delta_pos, delta_score = np.zeros(DIM), -float('inf')

# 4-6. MAIN GWO ITERATIVE LOOP
for t in range(max_iter):
    # Step 4: Evaluate fitness & rank Alpha, Beta, Delta
    for i in range(N):
        wolves[i] = np.clip(wolves[i], LB, UB) # Enforce bounds
        fitness = evaluate_fitness(wolves[i])

        if fitness > alpha_score:
            delta_score, delta_pos = beta_score, beta_pos.copy()
            beta_score, beta_pos = alpha_score, alpha_pos.copy()
            alpha_score, alpha_pos = fitness, wolves[i].copy()
        elif fitness > beta_score:
            delta_score, delta_pos = beta_score, beta_pos.copy()
            beta_score, beta_pos = fitness, wolves[i].copy()
        elif fitness > delta_score:
            delta_score, delta_pos = fitness, wolves[i].copy()

    # Step 6: Linearly decrease parameter 'a' from 2 to 0
    a = 2.0 - t * (2.0 / max_iter)

    # Step 5: Update positions of all wolves based on Alpha, Beta, Delta
    for i in range(N):
        for d in range(DIM):
            # Guidance from Alpha (Best)
            r1, r2 = np.random.rand(), np.random.rand()
            A1 = 2 * a * r1 - a
            C1 = 2 * r2
            D_alpha = abs(C1 * alpha_pos[d] - wolves[i, d])
            X1 = alpha_pos[d] - A1 * D_alpha

            # Guidance from Beta (2nd Best)
            r1, r2 = np.random.rand(), np.random.rand()
            A2 = 2 * a * r1 - a
            C2 = 2 * r2
            D_beta = abs(C2 * beta_pos[d] - wolves[i, d])
            X2 = beta_pos[d] - A2 * D_beta

            # Guidance from Delta (3rd Best)
            r1, r2 = np.random.rand(), np.random.rand()
            A3 = 2 * a * r1 - a
            C3 = 2 * r2
            D_delta = abs(C3 * delta_pos[d] - wolves[i, d])
            X3 = delta_pos[d] - A3 * D_delta

            # Average position update
            wolves[i, d] = (X1 + X2 + X3) / 3.0

    print(f"Iter {t+1:02d}/{max_iter} | Best Val F1-Score (Alpha): {alpha_score:.4f}")

# 7. OUTPUT THE BEST SOLUTION
best_C = 10 ** alpha_pos[0]
best_gamma = 10 ** alpha_pos[1]

final_svm = SVC(C=best_C, gamma=best_gamma, kernel='rbf', random_state=42).fit(X_train, y_train)
train_acc = accuracy_score(y_train, final_svm.predict(X_train))
val_acc = accuracy_score(y_val, final_svm.predict(X_val))

print("\n--- Optimal SVM Hyperparameters Found ---")
print(f"Best C            : {best_C:.4f}")
print(f"Best gamma        : {best_gamma:.6f}")
print(f"Validation F1     : {alpha_score:.4f}")
print(f"Train vs Val Acc  : {train_acc:.4f} vs {val_acc:.4f} (Controlled overfitting)")

Iter 01/20 | Best Val F1-Score (Alpha): 0.9862
Iter 02/20 | Best Val F1-Score (Alpha): 0.9908
Iter 03/20 | Best Val F1-Score (Alpha): 0.9908
Iter 04/20 | Best Val F1-Score (Alpha): 0.9908
Iter 05/20 | Best Val F1-Score (Alpha): 0.9908
Iter 06/20 | Best Val F1-Score (Alpha): 0.9908
Iter 07/20 | Best Val F1-Score (Alpha): 0.9908
Iter 08/20 | Best Val F1-Score (Alpha): 0.9908
Iter 09/20 | Best Val F1-Score (Alpha): 0.9908
Iter 10/20 | Best Val F1-Score (Alpha): 0.9908
Iter 11/20 | Best Val F1-Score (Alpha): 0.9908
Iter 12/20 | Best Val F1-Score (Alpha): 0.9908
Iter 13/20 | Best Val F1-Score (Alpha): 0.9908
Iter 14/20 | Best Val F1-Score (Alpha): 0.9908
Iter 15/20 | Best Val F1-Score (Alpha): 0.9908
Iter 16/20 | Best Val F1-Score (Alpha): 0.9908
Iter 17/20 | Best Val F1-Score (Alpha): 0.9908
Iter 18/20 | Best Val F1-Score (Alpha): 0.9908
Iter 19/20 | Best Val F1-Score (Alpha): 0.9908
Iter 20/20 | Best Val F1-Score (Alpha): 0.9908

--- Optimal SVM Hyperparameters Found ---
Best C           